# 02 — Parsing

DocFlow's 4 parsers and how they extract text + bounding boxes from documents.

## From pixels to structured text

Before an LLM can extract data from a document, something needs to convert the raw file into text. This is the parser's job, and it is more important than it looks — the quality of parsing directly determines the quality of extraction.

A parser produces three things:
1. **Text blocks** — chunks of text found on each page, in reading order.
2. **Bounding boxes** — the exact `(x0, y0, x1, y1)` coordinates of each block on the page, in PDF points (1 point = 1/72 inch). These are the foundation for evidence grounding: when DocFlow says a value was found at a specific location, it is because the parser placed a block there.
3. **Page metadata** — dimensions, block count, and the full raw text for each page.

The challenge is that PDFs come in two fundamentally different forms:

- **Digital (native) PDFs** have an embedded text layer. Parsers like PyMuPDF can extract text directly from the file structure in milliseconds, with pixel-perfect bounding boxes.
- **Scanned PDFs** are images wrapped in a PDF container — there is no text layer. These require OCR (optical character recognition) to recover the text. OCR is slower, less accurate, and the bounding boxes are approximate.

The **Smart parser** solves the "I don't know what kind of PDF this is" problem: it tries PyMuPDF first, checks if each page has enough usable text, and falls back to Tesseract OCR for pages that don't. This per-page decision means a document with some digital pages and some scanned pages gets the best parser for each.

In [ ]:
PDF_PATH = "data/sample_invoice.pdf"

## Parser Overview

DocFlow ships 4 parsers. All produce structured `Page` and `Block` objects with bounding boxes.

| Parser | Best for | Speed |
|---|---|---|
| `PyMuPDFParser` | Digital / native PDFs | ~100 ms/page |
| `TesseractParser` | Scanned / image-based PDFs | 1-5 s/page |
| `DoclingParser` | Complex tables and layouts | 4-5 s/page |
| `SmartParser` | Mixed documents (auto per-page) | Varies |

The smart parser tries PyMuPDF first, then falls back to Tesseract OCR for pages that lack usable text.

## PyMuPDF Parser

The fastest parser -- ideal for digital PDFs where text is embedded (not scanned).

In [ ]:
from docflow.parsing.pymupdf import PyMuPDFParser
from docflow.workflow.pipeline import Pipeline
from docflow.workflow.steps import Ingest, Parse

parser = PyMuPDFParser()
pipeline = Pipeline([Ingest(path=PDF_PATH), Parse(parser=parser)])
result = pipeline.run_sync()
document = result.state.document

print(f"Pages        : {len(document.pages)}")
print(f"Total blocks : {sum(p.block_count for p in document.pages)}")
print(f"Status       : {document.status}")
print()
print("--- raw_text preview (first 500 chars) ---")
print(document.raw_text[:500])

## Exploring Pages and Blocks

Each page contains a list of `Block` objects. Every block has a `text`, `block_type`, and an optional `BoundingBox` with coordinates on the page.

In [ ]:
for page in document.pages:
    print(f"=== Page {page.page_number} === ({page.block_count} blocks, {page.width:.0f}x{page.height:.0f} pts)")
    for i, block in enumerate(page.blocks[:5]):
        text_preview = block.text[:60].replace("\n", " ")
        print(f"  Block {i}: [{block.block_type.value}] \"{text_preview}\"")
        if block.bbox:
            bb = block.bbox
            print(f"           bbox: x0={bb.x0:.1f}, y0={bb.y0:.1f}, x1={bb.x1:.1f}, y1={bb.y1:.1f}"
                  f"  ({bb.width:.1f}w x {bb.height:.1f}h)")
    if page.block_count > 5:
        print(f"  ... and {page.block_count - 5} more blocks")
    print()

## Block Types and Structure

Blocks are classified by type (text, image, title, etc.). Below is the distribution across the document.

In [ ]:
from collections import Counter

all_blocks = [b for p in document.pages for b in p.blocks]
type_counts = Counter(b.block_type.value for b in all_blocks)
text_lengths = [len(b.text) for b in all_blocks if b.text]

print("Block type distribution:")
max_count = max(type_counts.values())
for btype, count in type_counts.most_common():
    bar = "#" * int(count / max_count * 40)
    print(f"  {btype:<12} {count:>3}  {bar}")

print()
print(f"Text length stats across {len(text_lengths)} blocks:")
print(f"  Shortest : {min(text_lengths):>5} chars")
print(f"  Longest  : {max(text_lengths):>5} chars")
print(f"  Average  : {sum(text_lengths) / len(text_lengths):>5.0f} chars")
print(f"  Total    : {sum(text_lengths):>5} chars")

## Tesseract Parser

OCR-based parser for scanned or image-based PDFs. Each page is rendered as an image, then Tesseract extracts text with word-level bounding boxes. Slower than PyMuPDF but essential when there is no embedded text layer.

In [ ]:
from docflow.parsing.tesseract_parser import TesseractParser

tesseract = TesseractParser()
pipeline_tess = Pipeline([Ingest(path=PDF_PATH), Parse(parser=tesseract)])
result_tess = pipeline_tess.run_sync()
doc_tess = result_tess.state.document

tess_blocks = sum(p.block_count for p in doc_tess.pages)

print(f"Pages        : {len(doc_tess.pages)}")
print(f"Total blocks : {tess_blocks}")
print()
print(f"Comparison: PyMuPDF produced {sum(p.block_count for p in document.pages)} blocks, "
      f"Tesseract produced {tess_blocks} blocks")
print()
print("--- Tesseract raw_text preview (first 500 chars) ---")
print(doc_tess.raw_text[:500])

## Docling Parser

The highest-quality parser for documents with complex tables, multi-column layouts, and nested structures. Docling understands document structure at a semantic level -- it produces first-class `Table` objects with row/column headers and per-cell bounding boxes. Significantly slower than PyMuPDF but worth it when table extraction matters.

> **Note:** Docling requires `pip install docflow[docling]` and has heavy dependencies. The cell below is wrapped in a try/except so the notebook runs even if Docling is not installed.

In [ ]:
try:
    from docflow.parsing.docling_parser import DoclingParser

    docling = DoclingParser()
    pipeline_doc = Pipeline([Ingest(path=PDF_PATH), Parse(parser=docling)])
    result_doc = pipeline_doc.run_sync()
    doc_docling = result_doc.state.document

    docling_blocks = sum(p.block_count for p in doc_docling.pages)

    print(f"Pages        : {len(doc_docling.pages)}")
    print(f"Total blocks : {docling_blocks}")
    print()

    for page in doc_docling.pages:
        tables = page.tables if hasattr(page, "tables") else []
        print(f"Page {page.page_number}: {page.block_count} blocks, {len(tables)} table(s)")
        for table in tables:
            print(f"  Table: {table.row_count} rows x {table.col_count} cols")
            if table.row_count > 0 and table.col_count > 0:
                cell = table.cell_at(0, 0)
                print(f"    cell(0,0): \"{cell.text}\"")

    print()
    print(f"Comparison: PyMuPDF={sum(p.block_count for p in document.pages)}, "
          f"Tesseract={tess_blocks}, Docling={docling_blocks} blocks")

except ImportError:
    print("Docling is not installed. Install with: pip install docflow[docling]")
    print("Skipping Docling parser demo.")

## Smart Parser

The smart parser runs PyMuPDF first, then checks each page for usable text. Pages with too little text (scanned, garbled, image-heavy) are re-processed with Tesseract OCR.

In [ ]:
from docflow.parsing.smart_parser import SmartParser

smart = SmartParser()
pipeline2 = Pipeline([Ingest(path=PDF_PATH), Parse(parser=smart)])
result2 = pipeline2.run_sync()
doc2 = result2.state.document

smart_blocks = sum(p.block_count for p in doc2.pages)
pymupdf_blocks = sum(p.block_count for p in document.pages)

print(f"PyMuPDF blocks : {pymupdf_blocks}")
print(f"Smart blocks   : {smart_blocks}")
print()
if smart_blocks == pymupdf_blocks:
    print("Block counts match -- the smart parser detected this PDF as fully digital")
    print("and used PyMuPDF for all pages (no OCR fallback needed).")
else:
    print(f"Difference: {abs(smart_blocks - pymupdf_blocks)} blocks.")
    print("The smart parser used OCR for some pages.")

## Searching Within Parsed Documents

`search_document` finds text across all pages and returns hits with page numbers, bounding boxes, and surrounding context.

In [ ]:
from docflow.search import search_document

hits = search_document(document, "Meridian")

print(f"Query: \"Meridian\"")
print(f"Total hits: {hits.total_hits}")
print()
for i, hit in enumerate(hits.hits):
    print(f"  Hit {i + 1}:")
    print(f"    Page     : {hit.page_number}")
    print(f"    Text     : \"{hit.text}\"")
    print(f"    Context  : \"{hit.context}\"")
    if hit.bbox:
        bb = hit.bbox
        print(f"    BBox     : x0={bb.x0:.1f}, y0={bb.y0:.1f}, x1={bb.x1:.1f}, y1={bb.y1:.1f}")
    print()

## Screenshots

Render PDF pages as PNG images. Useful for review UIs or visual debugging of bounding boxes.

In [ ]:
from docflow.screenshots import screenshot_pages_sync

shots = screenshot_pages_sync(file_path=PDF_PATH, output_dir="./output/pages", dpi=150)
print(f"Rendered {len(shots)} page(s)")
for s in shots:
    print(f"  Page {s.page_number}: {s.width}x{s.height} px  ->  {s.file_path}")

## Summary

Use **PyMuPDF** for digital PDFs where speed matters, **Tesseract** for scanned documents, and **Docling** when you need structured table extraction. When you are not sure what kind of PDF you have, use **SmartParser** -- it picks the right strategy per page automatically.